# Purged walk-forward models

Expanding-window folds keep repeated `(chain, token, support_anchor)` groups together, purge label windows that reach the test period, and preserve 24-hour gaps and embargoes. The shuffled split is retained only as a leakage contrast.

Logistic regularization, probability thresholds, calibration, and confirmation screens are selected from training history only.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("research/ml")
sys.path.insert(0, str(ROOT))
SNAPSHOT = ROOT / "data" / "snapshot.csv"
from scanner_ml.data import load_dataset
from scanner_ml.evaluate import classification_metrics
from scanner_ml.models import out_of_fold_predictions
from scanner_ml.validation import walk_forward_splits
if not SNAPSHOT.exists():
    print("Awaiting production snapshot; run npm run research:export")
else:
    data = load_dataset(SNAPSHOT)
    folds = walk_forward_splits(data)
    oof, details = out_of_fold_predictions(data, folds)
    rows = []
    for fold, part in oof.dropna(subset=["fold"]).groupby("fold"):
        for model in ("pred_logistic", "pred_hgb"):
            usable = part.dropna(subset=[model])
            if len(usable): rows.append({"fold": int(fold), "model": model, **classification_metrics(usable.y, usable[model])})
    display(rows)
    display(details)
